In [ ]:
# List of required packages
required_packages <- c( 
    "dplyr", 
    "tidyr",
    "data.table",
    "ggplot2",
    "readxl",
    "lubridate",
    "stringi",
    "broom",
    "scales",
    "bit64"
)

# Function to check and install missing packages
install_and_load <- function(packages) {
  installed_packages <- rownames(installed.packages())
  for (pkg in packages) {
    if (!pkg %in% installed_packages) {
      install.packages(pkg, dependencies = TRUE)
    }
    suppressMessages(library(pkg, character.only = TRUE))
  }
}

# Install and load the required packages
install_and_load(required_packages)

In [ ]:
# Define the file name
file_names <- c(
    "gp_clinical.txt", "gp_scripts.txt", "gp_registrations.txt", 
    "all_lkps_maps_v4.xlsx"
)

# Check if the file exists in the current directory
for (file in file_names) {
    if (!file.exists(file)) {
      message("File not found. Downloading from DNAnexus...")

      # Execute the dx download command
      system(paste("dx download project-XXXXXXXXXXXXXXXXXXXXXXXX:/", file, sep = ""))
    }
}

# Load the file as a tab-separated dataframe
gp_clinical <- fread("gp_clinical.txt", sep = "\t", header = TRUE)
gp_scripts <- fread("gp_scripts.txt", sep = "\t", header = TRUE)
gp_registrations <- fread("gp_registrations.txt", sep = "\t", header = TRUE)
mapping <- read_excel("all_lkps_maps_v4.xlsx", sheet = 14)

In [ ]:
# Define the file name
file_names <- c(
    "ukb_covariates_formatted.csv.gz"
)

# Check if the file exists in the current directory
for (file in file_names) {
    if (!file.exists(file)) {
      message("File not found. Downloading from DNAnexus...")

      # Execute the dx download command
      system(paste("dx download project-XXXXXXXXXXXXXXXXXXXXXXXX:/path/to/scratch/", file, sep = ""))
    }
}

# Load the file as a tab-separated dataframe
ukb_cov <- fread("ukb_covariates_formatted.csv.gz", sep = ",", header = TRUE)

In [ ]:
# TEMPORARY - REMOVE WHEN SCRIPT IS COMPLETE
# eid_list <- unique(gp_registrations$eid)[1:1000]
# gp_clinical <- gp_clinical[gp_clinical$eid %in% eid_list, ]
# gp_scripts <- gp_scripts[gp_scripts$eid %in% eid_list, ]
# gp_registrations <- gp_registrations[gp_registrations$eid %in% eid_list, ]

In [ ]:
gp_eid_list <- unique(gp_registrations$eid)
ukb_cov <- ukb_cov %>% filter(eid %in% gp_eid_list)

In [ ]:
# ============================================================
# Baseline time-zero trial (UKB recruitment date)
# Adds LDL0 = closest *pre-baseline* GP LDL (44P6.) before t0
# - DOES NOT restrict to those with GP LDL0
# - If no eligible pre-baseline GP LDL, LDL0 = NA
# ============================================================

parse_dmy <- function(x) {
  x <- fifelse(is.na(x) | x == "", NA_character_, x)
  dmy(x, quiet = TRUE)
}
to_num <- function(x) suppressWarnings(as.numeric(gsub("[^0-9.\\-]", "", x)))

# Statins (subset of scripts)
flag_statins <- function(gp_scripts) {
  scr <- as.data.table(copy(gp_scripts))
  scr[, issue_date := parse_dmy(issue_date)]
  scr <- scr[!is.na(issue_date)]

  scr[, bnf_clean := gsub("\\s+", "", bnf_code)]
  scr[, is_statin_bnf := !is.na(bnf_clean) & bnf_clean != "" &
    (startsWith(bnf_clean, "02.12.1") | startsWith(bnf_clean, "2.12.1"))]

  statin_regex <- "(atorvastatin|simvastatin|rosuvastatin|pravastatin|fluvastatin|lovastatin|pitavastatin)"
  scr[, is_statin_name := !is.na(drug_name) &
    stri_detect_regex(stri_trans_tolower(drug_name), statin_regex)]

  scr[, is_statin := is_statin_bnf | is_statin_name]
  scr[is_statin == TRUE, .(eid, issue_date)]
}

# LDL events (GP)
flag_ldl_44P6 <- function(gp_clinical,
                          ldl_code = "44P6.",
                          value_col = "value1",
                          min_event_date = as.Date("1995-01-01")) {
  cli <- as.data.table(copy(gp_clinical))
  cli[, event_dt := parse_dmy(event_dt)]
  cli <- cli[!is.na(event_dt)]
  cli <- cli[event_dt >= min_event_date]

  cli[, read_2_clean := stri_trim_both(read_2)]
  cli[, read_3_clean := stri_trim_both(read_3)]
  cli[, is_ldl := (read_2_clean == ldl_code | read_3_clean == ldl_code)]
  cli <- cli[is_ldl == TRUE]

  cli[, ldl_value := to_num(get(value_col))]
  cli <- cli[!is.na(ldl_value)]

  cli[, .(eid, event_dt, ldl_value)]
}

# Observable window from ALL scripts (not just statins)
derive_observable_from_scripts <- function(gp_scripts, min_date = as.Date("1995-01-01")) {
  scr <- as.data.table(copy(gp_scripts))
  scr[, issue_date := parse_dmy(issue_date)]
  scr <- scr[!is.na(issue_date)]
  scr <- scr[issue_date >= min_date]
  scr[, .(obs_start = min(issue_date), obs_end = max(issue_date)), by = eid]
}

build_baseline_trial_with_prebaseline_LDL0 <- function(cov,
                                                       gp_scripts,
                                                       gp_clinical,
                                                       min_event_date = as.Date("1995-01-01"),
                                                       washout_days = 365L,
                                                       grace_days = 180L,
                                                       ldl_code = "44P6.",
                                                       ldl_value_col = "value1",
                                                       ldl_min = 1,
                                                       ldl_max = 8,
                                                       one_row_per_eid = TRUE) {

  empty_cohort <- function() {
    data.table(
      eid = integer(),
      t0 = as.Date(character()),
      LDL0 = numeric(),
      LDL0_date = as.Date(character()),
      A = integer(),
      init_date = as.Date(character()),
      t_landmark = as.Date(character()),
      obs_start = as.Date(character()),
      obs_end = as.Date(character())
    )
  }

  counts <- data.table(step = character(), n = integer())
  add_count <- function(step, n) counts <<- rbind(counts, data.table(step = step, n = as.integer(n)))

  # ---- baseline cov ----
  cv <- as.data.table(copy(cov))
  cv[, t0 := as.Date(date_assessment)]
  cv <- cv[!is.na(eid) & !is.na(t0)]
  add_count("cov_rows_with_t0", nrow(cv))

  # ---- Observable window from ALL scripts (also defines "in primary care scripts") ----
  obs <- derive_observable_from_scripts(gp_scripts, min_date = min_event_date)
  setkey(obs, eid)
  add_count("obs_eids", nrow(obs))

  # Restrict to those with any scripts data (avoid treating missing as clean washout)
  cv <- obs[cv, on = "eid", nomatch = 0L]
  add_count("cov_in_primary_care_scripts", nrow(cv))
  if (nrow(cv) == 0) return(list(cohort = empty_cohort(), diagnostics = list(counts = counts)))

  # ---- Statins ----
  statin <- flag_statins(gp_scripts)
  setkey(statin, eid)
  add_count("statin_scripts", nrow(statin))

  # ---- Attach closest PRE-baseline GP LDL (do not restrict if missing) ----
  ldl <- flag_ldl_44P6(gp_clinical, ldl_code = ldl_code, value_col = ldl_value_col,
                       min_event_date = min_event_date)
  add_count("gp_ldl_events_after_min_date", nrow(ldl))

  if (nrow(ldl) > 0) {
    ldl <- ldl[ldl_value >= ldl_min & ldl_value <= ldl_max]
    add_count("gp_ldl_events_in_range", nrow(ldl))

    setnames(ldl, old = c("event_dt", "ldl_value"), new = c("LDL0_date", "LDL0"))
    setkey(ldl, eid)

    # Join on eid, then filter to LDL0_date < t0, then pick closest (max date)
    ldl_join <- ldl[cv, on = "eid", nomatch = 0L, .(eid, t0, LDL0_date, LDL0)]
    ldl_join <- ldl_join[LDL0_date < t0]

    if (nrow(ldl_join) > 0) {
      setorder(ldl_join, eid, t0, -LDL0_date)
      ldl_pre <- ldl_join[, .SD[1], by = .(eid, t0)]  # closest pre-baseline
    } else {
      ldl_pre <- data.table(eid = integer(), t0 = as.Date(character()),
                            LDL0_date = as.Date(character()), LDL0 = numeric())
    }

    cv <- merge(cv, ldl_pre, by = c("eid", "t0"), all.x = TRUE)
    add_count("with_LDL0_nonmissing", sum(!is.na(cv$LDL0)))
  } else {
    add_count("gp_ldl_events_in_range", 0L)
    cv[, `:=`(LDL0 = as.numeric(NA), LDL0_date = as.Date(NA))]
    add_count("with_LDL0_nonmissing", 0L)
  }
  add_count("with_LDL0_attached", nrow(cv))

  # Sanity: if LDL0 is NA then LDL0_date should be NA
  # (if this fails, there is a bug upstream in flag_ldl_44P6 / to_num parsing)
  stopifnot(cv[is.na(LDL0) & !is.na(LDL0_date), .N] == 0L)
  stopifnot(cv[!is.na(LDL0_date), all(LDL0_date < t0)])

  # ---- Eligibility: observable washout + grace around baseline ----
  cv[, ok_obs := (obs_start <= (t0 - days(washout_days))) &
                 (obs_end   >= (t0 + days(grace_days)))]
  idx <- cv[ok_obs == TRUE]
  add_count("ok_observable", nrow(idx))
  if (nrow(idx) == 0) return(list(cohort = empty_cohort(), diagnostics = list(counts = counts)))

  # Precompute join bounds (non-equi join needs columns)
  idx[, wash_start := t0 - days(washout_days)]
  idx[, wash_end   := t0]                  # exclusive
  idx[, grace_end  := t0 + days(grace_days)]

  # ---- Washout: no statins in (t0-washout, t0) ----
  statin_wash <- statin[idx,
                        on = .(eid, issue_date > wash_start, issue_date < wash_end),
                        nomatch = 0L,
                        .(eid, t0)][, .(has_statin_washout = TRUE), by = .(eid, t0)]

  idx <- merge(idx, statin_wash, by = c("eid", "t0"), all.x = TRUE)
  idx[is.na(has_statin_washout), has_statin_washout := FALSE]
  idx <- idx[has_statin_washout == FALSE]
  add_count("pass_washout", nrow(idx))
  if (nrow(idx) == 0) return(list(cohort = empty_cohort(), diagnostics = list(counts = counts)))

  # ---- Initiation window: any statin in (t0, t0+grace] (change if you want) ----
  statin_grace <- statin[idx,
                         on = .(eid, issue_date > t0, issue_date <= grace_end),
                         nomatch = 0L,
                         .(eid, t0, issue_date)]
  init_date <- statin_grace[, .(init_date = min(issue_date)), by = .(eid, t0)]

  idx <- merge(idx, init_date, by = c("eid", "t0"), all.x = TRUE)
  idx[, A := as.integer(!is.na(init_date))]
  idx[, t_landmark := grace_end]

  if (one_row_per_eid) {
    setorder(idx, eid, t0)
    idx <- idx[, .SD[1], by = eid]
  }
  add_count("final_rows", nrow(idx))

  cohort <- idx[, .(eid, t0, LDL0, LDL0_date, A, init_date, t_landmark, obs_start, obs_end)]
  list(cohort = cohort[order(eid, t0)], diagnostics = list(counts = counts))
}

In [ ]:
out <- build_baseline_trial_with_prebaseline_LDL0(
  cov = ukb_cov,
  gp_scripts = gp_scripts,
  gp_clinical = gp_clinical,
  min_event_date = as.Date("1990-01-01"),
  washout_days = 365,
  grace_days = 365,
  one_row_per_eid = TRUE
)
cohort <- out$cohort

cohort[, table(A, useNA="ifany")]
cohort[, summary(LDL0)]
out$diagnostics$counts

# quick checks
cohort[is.na(LDL0) & !is.na(LDL0_date), .N]   # should be 0
cohort[!is.na(LDL0_date), all(LDL0_date < t0)]# should be TRUE

In [ ]:
read2_icd10 <- read_excel("all_lkps_maps_v4.xlsx", sheet = 12)
read3_icd10 <- read_excel("all_lkps_maps_v4.xlsx", sheet = 17)

read2_icd10 <- as.data.table(read2_icd10)[!is.na(read_code) & !is.na(icd10_code)]
read3_icd10 <- as.data.table(read3_icd10)[!is.na(read_code) & !is.na(icd10_code)]

read2_icd10[, `:=`(read_code = stri_trim_both(as.character(read_code)),
                   icd10_code = stri_trim_both(as.character(icd10_code)))]
read3_icd10[, `:=`(read_code = stri_trim_both(as.character(read_code)),
                   icd10_code = stri_trim_both(as.character(icd10_code)))]

icd10_codes <- c("I20","I21","I22","I23","I24","I25")

read2_codes <- unique(read2_icd10[startsWith(icd10_code, icd10_codes), read_code])
read3_codes <- unique(read3_icd10[startsWith(icd10_code, icd10_codes), read_code])

In [ ]:
flag_events_by_codes <- function(gp_clinical,
                                 codes_read2 = character(),
                                 codes_read3 = character(),
                                 min_event_date = as.Date("1995-01-01")) {
  cli <- as.data.table(copy(gp_clinical))
  cli[, event_dt := parse_dmy(event_dt)]
  cli <- cli[!is.na(event_dt) & event_dt >= min_event_date]

  cli[, read_2_clean := stri_trim_both(as.character(read_2))]
  cli[, read_3_clean := stri_trim_both(as.character(read_3))]

  cli[, hit :=
        (length(codes_read2) > 0 & !is.na(read_2_clean) & read_2_clean %in% codes_read2) |
        (length(codes_read3) > 0 & !is.na(read_3_clean) & read_3_clean %in% codes_read3)
  ]

  cli[hit == TRUE, .(eid, event_dt, read_2 = read_2_clean, read_3 = read_3_clean)]
}

augment_with_prior_cad <- function(df,
                                       gp_clinical,
                                       read2_codes,
                                       read3_codes,
                                       min_event_date = as.Date("1995-01-01")) {

  coh <- as.data.table(copy(df))
  stopifnot(all(c("eid","t0") %in% names(coh)))
  coh[, t0 := as.Date(t0)]
  setkey(coh, eid)

  # initialise
  coh[, `:=`(
    prior_CAD_dt_latest = as.Date(NA),
    prior_CAD = factor("No prior CAD", levels = c(
      "No prior CAD",
      "Prior CAD within 180 days",
      "Prior CAD from 180 days - 2 years",
      "Prior CAD from 2 - 5 years",
      "Prior CAD from 5+ years"
    ))
  )]

  cad_ev <- flag_events_by_codes(
    gp_clinical,
    codes_read2 = read2_codes,
    codes_read3 = read3_codes,
    min_event_date = min_event_date
  )
  if (nrow(cad_ev) == 0) return(coh[])

  cad_ev <- as.data.table(cad_ev)
  cad_ev[, event_dt := as.Date(event_dt)]
  cad_ev <- cad_ev[!is.na(event_dt)]
  setkey(cad_ev, eid)

  # join all events per eid to each cohort row and keep only pre-t0
  tmp <- cad_ev[coh, on = "eid", allow.cartesian = TRUE][event_dt < t0]
  if (nrow(tmp) == 0) return(coh[])

  # latest CAD before t0 for each (eid,t0)
  prior_latest <- tmp[, .(prior_CAD_dt_latest = max(event_dt)), by = .(eid, t0)]
  setkey(prior_latest, eid, t0)
  setkey(coh, eid, t0)

  coh[prior_latest, prior_CAD_dt_latest := i.prior_CAD_dt_latest]

  # time since latest CAD in days
  coh[, cad_days_ago := as.numeric(t0 - prior_CAD_dt_latest)]

  # bin into requested categories (boundaries: <180, [180,2y), [2y,5y), >=5y)
  coh[, prior_CAD := fifelse(
    is.na(prior_CAD_dt_latest), "No prior CAD",
    fifelse(cad_days_ago < 180, "Prior CAD within 180 days",
      fifelse(cad_days_ago < 2*365, "Prior CAD from 180 days - 2 years",
        fifelse(cad_days_ago < 5*365, "Prior CAD from 2 - 5 years",
                "Prior CAD from 5+ years")
      )
    )
  )]

  coh[, prior_CAD := factor(prior_CAD, levels = c(
    "No prior CAD",
    "Prior CAD within 180 days",
    "Prior CAD from 180 days - 2 years",
    "Prior CAD from 2 - 5 years",
    "Prior CAD from 5+ years"
  ))]

  coh[, cad_days_ago := NULL]  # drop helper unless you want it
  coh[]
}

In [ ]:
cohort <- augment_with_prior_cad(
  df = cohort,                # must include eid and t_ldl0
  gp_clinical = gp_clinical,
  read2_codes = read2_codes,
  read3_codes = read3_codes
)

table(cohort$prior_CAD, useNA="ifany")

In [ ]:
augment_with_prior_statin <- function(df,
                                         gp_scripts,
                                         min_script_date = as.Date("1995-01-01"),
                                         washout_days = 365L) {

  levs <- c(
    "No prior statin",
    "1-2 years prior",
    "2-5 years prior",
    "5+ years prior"
  )

  coh <- as.data.table(copy(df))
  stopifnot(all(c("eid", "t0") %in% names(coh)))
  coh[, t0 := as.Date(t0)]
  stopifnot(inherits(coh$t0, "Date"))

  # initialise
  coh[, `:=`(
    prior_statin_dt_latest = as.Date(NA),
    prior_statin = factor("No prior statin", levels = levs)
  )]

  # statin scripts (dates)
  statin <- flag_statins(gp_scripts)
  if (nrow(statin) == 0) return(coh[])
  statin[, issue_date := as.Date(issue_date)]
  statin <- statin[!is.na(issue_date) & issue_date >= min_script_date]
  if (nrow(statin) == 0) return(coh[])

  setkey(statin, eid)
  setkey(coh, eid)

  # join all statin scripts for eid onto cohort rows; keep strictly pre-t0
  tmp <- statin[coh, on = "eid", allow.cartesian = TRUE][issue_date < t0]
  if (nrow(tmp) == 0) return(coh[])

  # latest pre-t0 statin per (eid,t0)
  prior_latest <- tmp[, .(prior_statin_dt_latest = max(issue_date)), by = .(eid, t0)]
  setkey(prior_latest, eid, t0)
  setkey(coh, eid, t0)
  coh[prior_latest, prior_statin_dt_latest := i.prior_statin_dt_latest]

  # days since last statin
  coh[, statin_days_ago := as.numeric(t0 - prior_statin_dt_latest)]

  # Because you already enforce a 12-month washout in cohort construction,
  # anything within < washout_days should not appear; if it does, set to NA and warn.
  if (coh[!is.na(statin_days_ago) & statin_days_ago < washout_days, .N] > 0) {
    warning("Found prior statin scripts within washout window; check cohort washout logic.")
  }

  # bin: 1-2y, 2-5y, 5+y, else No prior statin (including NA)
  coh[, prior_statin := fifelse(
    is.na(prior_statin_dt_latest), "No prior statin",
    fifelse(statin_days_ago < 2*365, "1-2 years prior",
      fifelse(statin_days_ago < 5*365, "2-5 years prior",
              "5+ years prior")
    )
  )]

  coh[, prior_statin := factor(prior_statin, levels = levs)]
  coh[, statin_days_ago := NULL]

  coh[]
}

In [ ]:
cohort <- augment_with_prior_statin(
  df = cohort,                # must include eid and t_ldl0
  gp_scripts = gp_scripts,
  washout_days = 365L
)

table(cohort$prior_statin, useNA="ifany")

In [ ]:
augment_with_utilisation <- function(df,
                                     gp_scripts,
                                     gp_clinical,
                                     lookback_days = 365L,
                                     min_date = as.Date("1990-01-01"),
                                     suffix = NULL) {
  coh <- as.data.table(copy(df))
  stopifnot(all(c("eid","t0") %in% names(coh)))
  coh[, t0 := as.Date(t0)]
  coh[, lb_start := t0 - days(lookback_days)]

  # pick a suffix automatically if not provided
  if (is.null(suffix)) suffix <- paste0(lookback_days, "d")

  nm <- function(base) paste0(base, "_", suffix)

  # ---------- scripts utilisation ----------
  scr <- as.data.table(copy(gp_scripts))
  scr[, issue_date := parse_dmy(issue_date)]
  scr <- scr[!is.na(issue_date) & issue_date >= min_date]
  setkey(scr, eid)

  scr_j <- scr[coh, on = "eid", allow.cartesian = TRUE,
               .(eid, t0, lb_start, issue_date)]
  scr_j <- scr_j[issue_date >= lb_start & issue_date < t0]

  scr_u <- scr_j[, .(
    n_scripts = .N,
    n_rx_days = uniqueN(issue_date)
  ), by = .(eid, t0)]
  setnames(scr_u, c("n_scripts","n_rx_days"), c(nm("n_scripts"), nm("n_rx_days")))

  # ---------- clinical utilisation ----------
  cli <- as.data.table(copy(gp_clinical))
  cli[, event_dt := parse_dmy(event_dt)]
  cli <- cli[!is.na(event_dt) & event_dt >= min_date]
  setkey(cli, eid)

  cli_j <- cli[coh, on = "eid", allow.cartesian = TRUE,
               .(eid, t0, lb_start, event_dt)]
  cli_j <- cli_j[event_dt >= lb_start & event_dt < t0]

  cli_u <- cli_j[, .(
    n_clin_events = .N,
    n_clin_days   = uniqueN(event_dt)
  ), by = .(eid, t0)]
  setnames(cli_u, c("n_clin_events","n_clin_days"), c(nm("n_clin_events"), nm("n_clin_days")))

  # ---------- merge back; fill missing with 0 ----------
  out <- merge(coh, scr_u, by = c("eid","t0"), all.x = TRUE)
  out <- merge(out, cli_u, by = c("eid","t0"), all.x = TRUE)

  for (v in c(nm("n_scripts"), nm("n_rx_days"), nm("n_clin_events"), nm("n_clin_days"))) {
    if (!v %in% names(out)) out[, (v) := 0L]
    out[is.na(get(v)), (v) := 0L]
  }

  out[, lb_start := NULL]
  out[]
}

In [ ]:
cohort <- augment_with_utilisation(
  df = cohort,                # must include eid and t_ldl0
  gp_scripts = gp_scripts,
  gp_clinical = gp_clinical,
  lookback_days = 365L
)

mean(cohort$n_scripts, na.rm=TRUE)

In [ ]:
# ============================================================
# Augment landmark cohort with LDL1, keeping everyone.
# Adds:
#   - ldl1_end
#   - LDL1_dt, LDL1 (NA if unobserved)
#   - has_LDL1 (TRUE/FALSE)
# ============================================================
augment_with_ldl1_full <- function(cohort,
                                   gp_clinical,
                                   ldl_code = "44P6.",
                                   ldl_value_col = "value1",
                                   min_event_date = as.Date("1995-01-01"),
                                   ldl1_days = 365L,
                                   ldl_min = 1,
                                   ldl_max = 8,
                                   take = c("first", "closest_to_end")) {

  take <- match.arg(take)

  coh <- as.data.table(copy(cohort))
  stopifnot(all(c("eid", "t_landmark") %in% names(coh)))

  # Define LDL1 window end
  coh[, ldl1_end := t_landmark + days(ldl1_days)]

  # Pull LDL events
  ldl <- flag_ldl_44P6(gp_clinical, ldl_code = ldl_code, value_col = ldl_value_col,
                       min_event_date = min_event_date)
  ldl <- ldl[ldl_value >= ldl_min & ldl_value <= ldl_max]
  setkey(ldl, eid)

  # Candidates within window (only for those who have any)
  ldl1_cand <- ldl[coh,
                   on = .(eid, event_dt > t_landmark, event_dt <= ldl1_end),
                   allow.cartesian = TRUE,
                   nomatch = 0L,
                   .(eid, LDL1_dt = event_dt, LDL1 = ldl_value, ldl1_end)]

  if (nrow(ldl1_cand) == 0) {
    coh[, `:=`(LDL1_dt = as.Date(NA), LDL1 = NA_real_, has_LDL1 = FALSE)]
    return(coh[])
  }

  # Pick one LDL1 per eid
  if (take == "first") {
    setorder(ldl1_cand, eid, LDL1_dt)
    ldl1_pick <- ldl1_cand[, .SD[1], by = eid]
  } else {
    ldl1_cand[, dist_to_end := abs(as.integer(ldl1_end - LDL1_dt))]
    setorder(ldl1_cand, eid, dist_to_end, LDL1_dt)
    ldl1_pick <- ldl1_cand[, .SD[1], by = eid][, c("ldl1_end", "dist_to_end") := NULL]
  }

  # Merge onto full cohort
  coh <- merge(coh, ldl1_pick[, .(eid, LDL1_dt, LDL1)], by = "eid", all.x = TRUE)
  coh[, has_LDL1 := !is.na(LDL1_dt)]

  coh[]
}

In [ ]:
# ----------------------------
# Add a follow-up LDL measurement as an outcome
# ----------------------------
cohort_outcome <- augment_with_ldl1_full(cohort, gp_clinical, ldl1_days = 365, take = "first")
cohort_outcome[, table(A, has_LDL1, useNA="ifany")]

In [ ]:
dat_all <- cohort_outcome %>%
  mutate(
    A = as.integer(A),
    A_f = factor(A, levels = c(0,1), labels = c("No statin initiation", "Initiated statin")),
    dLDL = LDL1 - LDL0
  )

dat_ldl1 <- dat_all %>% filter(has_LDL1 == TRUE, !is.na(LDL1), !is.na(LDL0), !is.na(A))

# Colour-blind friendly palette (Okabe-Ito)
pal <- c("No statin initiation" = "#0072B2", "Initiated statin" = "#D55E00")

theme_pub <- theme_minimal(base_size = 12) +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    legend.title = element_blank(),
    legend.position = "top",
    plot.title.position = "plot",
    plot.title = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey25"),
    axis.title = element_text(face = "bold")
  )

# Helper: annotate mean and 95% CI for a vector
mc_ci_label <- function(x) {
  x <- x[is.finite(x)]
  m <- mean(x)
  lo <- quantile(x, 0.025)
  hi <- quantile(x, 0.975)
  sprintf("Mean %.2f\n95%% range [%.2f, %.2f]", m, lo, hi)
}

In [ ]:
p_has <- dat_all %>%
  group_by(A_f) %>%
  summarise(
    n = n(),
    n_ldl1 = sum(has_LDL1, na.rm = TRUE),
    prop = n_ldl1 / n,
    .groups = "drop"
  ) %>%
  ggplot(aes(x = A_f, y = prop, fill = A_f)) +
  geom_col(width = 0.65, alpha = 0.90) +
  geom_text(aes(label = percent(prop, accuracy = 0.1)),
            vjust = -0.4, size = 4.0) +
  scale_fill_manual(values = pal) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, NA)) +
  labs(
    title = "Follow-up LDL measurement frequency differs by initiation status",
    subtitle = "Proportion with at least one LDL measurement in the LDL1 window",
    x = NULL,
    y = "Proportion with LDL1 observed"
  ) +
  theme_pub

p_has

In [ ]:
p_hist <- dat_ldl1 %>%
  ggplot(aes(x = dLDL, fill = A_f)) +
  geom_histogram(aes(y = after_stat(density)),
                 bins = 40, alpha = 0.55, colour = "white", linewidth = 0.2) +
  geom_density(linewidth = 0.9, alpha = 0.0, aes(colour = A_f)) +
  scale_fill_manual(values = pal) +
  scale_colour_manual(values = pal) +
  labs(
    title = "Distribution of LDL change by statin initiation status",
    subtitle = "Among individuals with LDL1 observed",
    x = "LDL change: LDL1 − LDL0 (mmol/L)",
    y = "Density"
  ) +
  theme_pub

# Add group-specific summary text (mean and 95% range) to the histogram
sum_txt <- dat_ldl1 %>%
  group_by(A_f) %>%
  summarise(
    label = mc_ci_label(dLDL),
    .groups = "drop"
  ) %>%
  mutate(
    x = quantile(dat_ldl1$dLDL, probs = 0.02, na.rm = TRUE),
    y = Inf
  )

p_hist <- p_hist +
  geom_text(
    data = sum_txt,
    aes(x = x, y = y, label = label, colour = A_f),
    hjust = 0, vjust = 1.15, size = 3.3, lineheight = 1.1,
    show.legend = FALSE
  )

p_hist

In [ ]:
p_scatter <- dat_ldl1 %>%
  ggplot(aes(x = LDL0, y = LDL1, colour = A_f)) +
  geom_point(alpha = 0.25, size = 1.8) +
  geom_smooth(method = "lm", se = TRUE, linewidth = 1.0) +
  scale_colour_manual(values = pal) +
  labs(
    title = "LDL1 vs LDL0 by initiation status",
    subtitle = "Linear fit shown separately by group (descriptive)",
    x = "Baseline LDL0 (mmol/L)",
    y = "Follow-up LDL1 (mmol/L)"
  ) +
  theme_pub

p_scatter

In [ ]:
dat_A <- dat_all %>% filter(!is.na(LDL0), !is.na(A))

fit_logit <- glm(A ~ LDL0, data = dat_A, family = binomial())

grid <- tibble(LDL0 = seq(quantile(dat_A$LDL0, 0.01, na.rm = TRUE),
                         quantile(dat_A$LDL0, 0.99, na.rm = TRUE),
                         length.out = 200))
pred <- predict(fit_logit, newdata = grid, type = "link", se.fit = TRUE)
grid <- grid %>%
  mutate(
    p = plogis(pred$fit),
    p_lo = plogis(pred$fit - 1.96 * pred$se.fit),
    p_hi = plogis(pred$fit + 1.96 * pred$se.fit)
  )

p_logit <- ggplot() +
  geom_ribbon(data = grid, aes(x = LDL0, ymin = p_lo, ymax = p_hi), alpha = 0.15) +
  geom_line(data = grid, aes(x = LDL0, y = p), linewidth = 1.1) +
  geom_jitter(
    data = dat_A %>% sample_n(min(nrow(dat_A), 20000)),
    aes(x = LDL0, y = A),
    height = 0.03, width = 0.0, alpha = 0.05, size = 0.8
  ) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  labs(
    title = "Initiation probability increases with baseline LDL0",
    subtitle = "Logistic regression: A ~ LDL0 (with 95% CI band)",
    x = "Baseline LDL0 (mmol/L)",
    y = "Pr(initiate within 180 days)"
  ) +
  theme_pub

p_logit

In [ ]:
write.table(cohort_outcome, "ukb_gp_formatted_from_assessment.csv.gz", row.names=FALSE, sep=',', quote=FALSE)
system(paste("dx upload", "ukb_gp_formatted_from_assessment.csv.gz", "--path project-XXXXXXXXXXXXXXXXXXXXXXXX:/path/to/scratch/"))